# Information Retrieval using ChatGPT and the OpenAI Python API library

## Importing Libraries and preparing data

In [1]:
import openai
from openai import OpenAI
import pandas as pd

In [2]:
# Prepare the connection through the OpenAI Python API library. Credentials will be read from the environment file by default.
client = OpenAI()

In [3]:
FILE_PATH = f"../data/spotify_reviews_filtered.json"

# Fetch relevant data
try:
    df = pd.read_csv('../data/spotify_reviews.csv')
except Exception as e:
    raise Exception(f"Failed to read CSV file: {e}")

# Filter and format relevant data
df = df.drop(columns=['reviewId', 'userName', 'score', 'thumbsUpCount', 'reviewCreatedVersion'])
df['at'] = pd.to_datetime(df['at'])

# Filter data for relevant timeframe
df = df[df['at'] > '2023-11-01']
df = df[df['at'] < '2024-02-01']
 
# Save data as JSON file
try:
    df.to_json(FILE_PATH, orient='records')
except Exception as e:
    raise Exception(f"Failed to write JSON file: {e}")

## Setting up assistant

In [4]:
system_prompt = """
                You are a helpful assistant analyzing Spotify app reviews. 
                Be precise and focus on specific features which are highlighted in the reviews. 
                Provide 5 to 10 words to explain the particular feature for each prompt.
                Limit your response to the data available.
                Do not provide general information.
                Provide your response in a suitable markdown format.
                """

In [5]:
# Create the Assistant
print("Creating assistant...")
try:
    assistant = client.beta.assistants.create(
        name="Spotify Review Analyzer",
        instructions=system_prompt,
        model="gpt-3.5-turbo",
        tools=[{"type": "file_search"}],
        temperature=0.01
    )
except Exception as e:
    raise Exception(f"Failed to create assistant: {e}")

# Create a vector store to store the data
print("Creating vector store...")
try:
    vector_store = client.beta.vector_stores.create(name=f"Spotify Review Vector Store")
except Exception as e:
    raise Exception(f"Failed to create vector store: {e}")

# Prepare files for upload to OpenAI
file_paths = [FILE_PATH]
file_streams = [open(path, "rb") for path in file_paths]

# Use SDK helper to upload the files, add them to the vector store and poll the status of the file batch for completion.
print("Uploading files...")
try:
    file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=file_streams
)
except Exception as e:
    raise Exception(f"Failed to upload files: {e}")
    
# You can print the status and the file counts of the batch to see the result of this operation.
print("Finished uploading files.")
print(file_batch.status)
print(file_batch.file_counts)

# Update the assistant with the vector store
print("Updating assistant with vector store...")
try:
    assistant = client.beta.assistants.update(
        assistant_id=assistant.id,
        tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
    )
except Exception as e:
    raise Exception(f"Failed to update assistant: {e}")

Creating assistant...
Creating vector store...
Uploading files...
Finished uploading files.
completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
Updating assistant with vector store...


In [6]:
# Create the Thread
try:
    thread = openai.beta.threads.create()
except Exception as e:
    raise Exception(f"Failed to create thread: {e}")

# Add individual review to the thread
def add_user_message(message):
    try:
        openai.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=message
        )
    except Exception as e:
        raise Exception(f"Failed to add user message: {e}")

# Function to run the assistant on the thread
def run_assistant():
    try:
        run = openai.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id
        )
        return run
    except Exception as e:
        raise Exception(f"Failed to run assistant: {e}")

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    try:
        messages = openai.beta.threads.messages.list(
            thread_id=thread.id
        )
        response_message = messages.data[0]
        return response_message.content[0].text.value
    except Exception as e:
        raise Exception(f"Failed to get assistant response: {e}")

# Function to add user message and run the assistant
def add_user_message_and_run(user_prompt):
    add_user_message(user_prompt)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    return assistant_response

## Run Prompts to Get Results

In [7]:
prompt_likes = """
                Please extract the top 10 specific features that users like the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_likes))

Based on the reviews data, the top 10 specific features that users like the most about the Spotify app are:

1. Extensive music collection
2. Easy to use interface
3. Custom playlist options
4. Recommendations for new music
5. Ability to listen offline
6. Great selection of songs
7. Introduces users to new artists
8. Personalized playlists
9. Convenient device options
10. Fantastic application with tailored playlists【4:1†source】【4:2†source】【4:4†source】.


In [8]:
prompt_dislikes = """
                    Please extract the top 10 specific features that users dislike the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_dislikes))

Based on the reviews data, the top 10 specific features that users dislike the most about the Spotify app are:

1. Incessant ads and limited skips for free users
2. Lack of customization and poor navigation
3. Pushing expensive premium plans
4. Difficulty in removing unfollowed podcasts from the feed
5. Playing songs added by Spotify in user playlists
6. Changes in the like button functionality
7. Premium features limiting basic user controls
8. Inability to play songs in desired order without premium
9. Excessive ads disrupting music flow
10. Removal of basic features and excessive ads for free users【8:0†source】【8:1†source】【8:2†source】【8:4†source】.


In [9]:
prompt_wants = """
                Please extract the top 10 specific features that users want to see in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_wants))

Based on the reviews data, the top 10 specific features that users want to see in the Spotify app are:

1. Ability to play songs in desired order without premium
2. Option to sort/group liked songs by genre
3. Improved shuffle functionality for large playlists
4. Easier liking system for songs and albums
5. Enhanced share options for song lyrics
6. Repeat track button in the notification bar
7. Mass sorting of songs into playlists by genre
8. Ability to select songs without a premium subscription
9. Access to basic features without the need for premium
10. Improved free user experience with essential features available【12:0†source】【12:1†source】【12:4†source】.


In [12]:
prompt_bugs = """
                Please extract the top 10 specific bugs that users have reported in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_bugs))

The specific bugs that users have reported in the Spotify app include:

1. Songs listed multiple times in library, smart shuffle issues
2. App slowness and freezing on Android devices
3. Random removal of downloads, requiring constant redownloading
4. App crashes after ads, need for multiple restarts
5. Sound quality fixes not available in free version
6. Autoplay repetition of specific songs
7. Unreliable background playback, skipping issues
8. Liked songs not added to the correct playlist
9. Music freezing mid-playback, songs not progressing
10. Ads frequency and duration issues, interruptions in music playback【24:0†source】.
